# Live ALPSS box plot

Helpers live in `aimdl_examples.live`.

In [ ]:
import asyncio
import io
import sys
from pathlib import Path

import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

load_dotenv(find_dotenv())
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from aimdl_examples import get_girder_client
from aimdl_examples.live import (
    METRICS, TIMEFRAMES, VELOCITY,
    Config, Store,
    build_material_map, build_view, new_item_ids, poll_count,
    fetch_new_items, parse_many, render, seed_store,
)

session = requests.Session()
gc = get_girder_client(session=session)
print('connected:', gc.urlBase)

In [ ]:
cfg = Config(metric=VELOCITY, seed_igsns=12, window_size=10, poll_seconds=30)

# Replay mode: hold back the newest rows and feed them in, to exercise the live path
# without waiting on the lab. "New" is an arrival counter, not a clock, so this drives
# the exact same highlight code the live path runs.
SIMULATE = False
SIM_HOLDBACK = 40
SIM_ROWS_PER_TICK = 4

In [ ]:
store = Store(jitter=cfg.jitter)
material_map = build_material_map(gc)
print(f'material map: {len(material_map)} base IGSNs')

scanned = seed_store(gc, store, cfg)
last_count = poll_count(gc, cfg)
print(f'seeded {len(store.rows)} rows / {len(store.igsns(cfg.metric))} IGSNs '
      f'from {scanned} items | server count={last_count}')

pending_rows = []
if SIMULATE:
    # hold back the newest rows; un-see them so the replay re-admits them
    held = sorted(store.rows, key=lambda r: r['arrival_seq'])[-SIM_HOLDBACK:]
    keep = {id(r) for r in held}
    store.rows = [r for r in store.rows if id(r) not in keep]
    for row in held:
        store.seen.discard(row['itemId'])
    pending_rows = held
    print(f'holding back {len(pending_rows)} rows for replay')


def replay_tick():
    global pending_rows
    if not pending_rows:
        return 'replay exhausted'
    batch, pending_rows = pending_rows[:SIM_ROWS_PER_TICK], pending_rows[SIM_ROWS_PER_TICK:]
    return f'replayed +{store.add(batch)} ({len(pending_rows)} left)'

In [ ]:
w_metric = W.Dropdown(options=list(METRICS), value=cfg.metric, description='Metric')
w_highlight = W.Checkbox(value=True, description='Highlight new')
w_tf = W.Dropdown(options=list(TIMEFRAMES), value='last 24h', description='Timeframe')
w_x = W.Dropdown(options=['igsn', 'material_type'], value='igsn', description='x-axis')
w_size = W.IntSlider(value=cfg.window_size, min=3, max=25, description='Boxes')
w_follow = W.Checkbox(value=True, description='Follow live')
w_poll = W.BoundedIntText(value=cfg.poll_seconds, min=5, max=600, description='Poll (s)')
b_prev = W.Button(description='\u25c0 Older')
b_next = W.Button(description='Newer \u25b6')
b_stop = W.Button(description='Stop', button_style='danger')
status = W.HTML()

# An Image widget, NOT an Output widget. `with out: display(fig)` only captures while a
# cell is executing -- the async poll loop has no cell context, so every tick's figure
# escapes the widget and stacks in the cell's output area instead. Assigning
# Image.value is a comm message, so it lands in the widget from any context.
canvas = W.Image(format='png', layout=W.Layout(width='880px'))

window_offset = 0
fig, ax = plt.subplots(figsize=(max(6.0, 0.9 * cfg.window_size), 4.2))
fig.subplots_adjust(left=0.10, right=0.98, bottom=0.34, top=0.92)
plt.close(fig)   # never auto-shown; it only ever leaves via savefig below


def redraw(message=''):
    global window_offset
    if w_follow.value:
        window_offset = 0

    view = build_view(store, material_map, cfg, metric=w_metric.value, xcol=w_x.value,
                      timeframe=w_tf.value, window_size=w_size.value,
                      window_offset=window_offset)
    new_ids = new_item_ids(view.base, cfg.highlight_last_k) if w_highlight.value else set()
    render(ax, view, cfg, xcol=w_x.value, metric=w_metric.value,
           new_ids=new_ids, highlight=w_highlight.value)

    shown = set(view.rows['itemId']) if not view.rows.empty else set()
    ax.set_title(f'{view.stats["n_all"]} plottable | {view.stats["n_cats"]} {w_x.value} '
                 f'in timeframe | showing {len(view.categories)}', pad=8)

    notes = [message]
    outside = len(new_ids - shown)
    if outside:
        # Highlight keys on arrival; the window filters on experiment time. A result
        # ingested now can carry an old experiment_date. Say so; never silently drop.
        notes.append(f'<b style="color:#d62728">{outside} new outside window</b>')
    if view.stats['n_no_ts']:
        notes.append(f'{view.stats["n_no_ts"]} rows lack experiment_date')
    if window_offset:
        notes.append(f'scrolled back {window_offset}')
    status.value = ' &nbsp;|&nbsp; '.join(n for n in notes if n)

    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=96, bbox_inches='tight')
    canvas.value = buffer.getvalue()


def shift(delta):
    def handler(_):
        global window_offset
        w_follow.value = False
        view = build_view(store, material_map, cfg, metric=w_metric.value, xcol=w_x.value,
                          timeframe=w_tf.value, window_size=w_size.value, window_offset=0)
        ceiling = max(0, view.stats['n_cats'] - w_size.value)
        window_offset = int(np.clip(window_offset + delta, 0, ceiling))
        redraw()
    return handler


b_prev.on_click(shift(+1))
b_next.on_click(shift(-1))
for widget in (w_metric, w_highlight, w_tf, w_x, w_size, w_follow):
    widget.observe(lambda change: redraw(), names='value')

display(W.VBox([
    W.HBox([w_metric, w_highlight, w_tf, w_x]),
    W.HBox([w_size, b_prev, b_next, w_follow]),
    W.HBox([w_poll, b_stop]),
    status, canvas,
]))
redraw('seeded')

In [ ]:
# async, not `while True: time.sleep()`: a blocking sleep starves the kernel event loop
# and the widgets above go dead until it exits. Network work is pushed to an executor
# for the same reason.
running = False


async def tick():
    global last_count
    if SIMULATE:
        return replay_tick()

    loop = asyncio.get_event_loop()
    count = await loop.run_in_executor(None, lambda: poll_count(gc, cfg))
    if count == last_count:
        return f'no change (count={count})'

    items = await loop.run_in_executor(None, lambda: fetch_new_items(gc, cfg, store.seen))
    items = list(reversed(items))   # oldest-first, so arrival_seq ascends with ingestion
    rows = await loop.run_in_executor(None, lambda: parse_many(gc, items, cfg.max_workers))
    added, skipped, errors = store.ingest(items, rows)
    last_count = count

    problem = f' <b style="color:#d62728">{errors} err</b>' if errors else ''
    return f'+{added} new ({skipped} without any metric){problem} | count={count}'


async def run():
    global running
    running = True
    while running:
        try:
            message = await tick()
        except Exception as exc:
            message = f'<span style="color:#d62728">error: {type(exc).__name__}: {exc}</span>'
        redraw(message)
        await asyncio.sleep(int(w_poll.value))
    redraw('stopped')


b_stop.on_click(lambda _: globals().__setitem__('running', False))

# Re-running this cell must not leave the previous loop alive: two run() tasks means two
# redraws per tick, racing over the same figure.
try:
    task.cancel()
except NameError:
    pass
task = asyncio.ensure_future(run())
print('polling started -- scroll up for the plot')